# Hyperparameter impact exploration: baseline vs. recommended

This notebook loads two trained models and compares:
1. **Training curves** — ELBO, reconstruction loss, per-group KL
2. **Shared latent UMAPs** — coloured by cluster label, antigen specificity, and time point
3. **Integration metrics** — quantitative assessment of batch mixing and biological structure preservation

**Models compared:**
- `results/spvipes_bcells3` — baseline (original `malaria_bcells.ipynb`)
- `results/spvipes_bcells_recommended` — recommended hyperparameters (`malaria_bcells_recommended.ipynb`)

> Run `malaria_bcells.ipynb` and `malaria_bcells_recommended.ipynb` before this notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

import scvi
import torch
import matplotlib.pyplot as plt
import anndata as ad
import scipy.sparse as sp
import spVIPESmulti

import seaborn as sns
from pathlib import Path

np.random.seed(0)
torch.manual_seed(0)
sc.settings.set_figure_params(dpi=80, frameon=False)

print(f"spVIPESmulti  : {spVIPESmulti.__version__}")
print(f"scvi-tools: {scvi.__version__}")
print(f"scanpy   : {sc.__version__}")
print(f"torch    : {torch.__version__} (CUDA available: {torch.cuda.is_available()})")
print(f"anndata  : {ad.__version__}")

spVIPESmulti  : 1.0.0
scvi-tools: 1.4.2
scanpy   : 1.12.1
torch    : 2.11.0+cu128 (CUDA available: True)
anndata  : 0.12.11


In [2]:
root = Path("/exports/para-lipg-hpc/mdmanurung/spVIPESmulti/docs/notebooks/")
path_obs = root/"data/bcells_obs.csv"
path_rna = root/"data/bcells_rna.csv"

In [3]:
adata = ad.read_csv(path_rna, first_column_names=True)
obs = pd.read_csv(path_obs, index_col=0)
adata.obs = obs
adata.layers["counts"] = adata.X.copy()

In [4]:
adata.obs["antigen_class"] = (
    adata.obs["antigen_specific"]
    .astype(str)
    .str.strip()
    .apply(lambda x: "Negative" if x.lower() == "negative" else "Positive")
)
adata.obs["antigen_class"].value_counts()
adata = adata[adata.obs["antigen_class"] == "Positive"]
adata

View of AnnData object with n_obs × n_vars = 9978 × 38605
    obs: 'nCount_RNA', 'nFeature_RNA', 'nCount_ADT', 'nFeature_ADT', 'nCount_HTO', 'nFeature_HTO', 'batch', 'cellhashr_id', 'percent_mito', 'percent_ribo', 'percent_mito_ribo', 'log10GenesPerUMI', 'percent_top50', 'percent_hemo', 'scdblfinder_score', 'scdblfinder_class', 'scran_size_factors', 'CTgene', 'CTnt', 'CTaa', 'CTstrict', 'clonalProportion', 'clonalFrequency', 'cloneSize', 'barcode', 'IGH', 'cdr3_aa1', 'cdr3_nt1', 'IGLC', 'cdr3_nt2', 'has_bcr', 'NANP', 'NJunction', 'qc_keep', 'RNA.weight', 'ADT.weight', 'wsnn_res.1', 'seurat_clusters', 'MG_nn_res.0.1', 'MG_res.0.1', 'MG_nn_res.0.2', 'MG_res.0.2', 'MG_nn_res.0.3', 'MG_res.0.3', 'MG_nn_res.0.4', 'MG_res.0.4', 'MG_nn_res.0.5', 'MG_res.0.5', 'MG_nn_res.0.6', 'MG_res.0.6', 'MG_nn_res.0.7', 'MG_res.0.7', 'MG_nn_res.0.8', 'MG_res.0.8', 'MG_nn_res.0.9', 'MG_res.0.9', 'MG_nn_res.1', 'MG_res.1', 'MG_nn_res.1.1', 'MG_res.1.1', 'MG_nn_res.1.2', 'MG_res.1.2', 'MG_nn_res.1.3', 'MG_res

In [5]:
sc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor="seurat_v3", batch_key="batch")
adata = adata[:, adata.var["highly_variable"]].copy()

In [6]:
# Split into per-time-point AnnData objects
antigen_specific = sorted(adata.obs["antigen_specific"].unique())
adatas_dict = {}
for ac in antigen_specific:
    sub = adata[adata.obs["antigen_specific"] == ac].copy()
    # Clean up — prepare_adatas doesn't need extra obsm/uns
    sub.uns = {}
    sub.obsm = {}
    sub.layers = {}
    adatas_dict[ac] = sub
    print(f"  {ac}: {sub.shape}")

# Concatenate with spVIPESmulti
adata_spv = spVIPESmulti.data.prepare_adatas(adatas_dict)

print(f"\nConcatenated AnnData: {adata_spv.shape}")
print(f"Groups             : {list(adata_spv.uns['groups_mapping'].values())}")
print(f"Group sizes        : {[len(g) for g in adata_spv.uns['groups_obs_indices']]}")

  CRXV: (6898, 3000)
  NANP: (2207, 3000)
  Njunc: (873, 3000)

Concatenated AnnData: (9978, 9000)
Groups             : ['CRXV', 'NANP', 'Njunc']
Group sizes        : [6898, 2207, 873]


In [7]:
spVIPESmulti.model.spVIPESmulti.setup_anndata(
    adata_spv,
    groups_key="antigen_specific",
    label_key="cluster_label",
    batch_key="batch"
)

INFO     spVIPESmulti: === spVIPESmulti AnnData Setup ===                                                          
INFO     spVIPESmulti: Setting up with groups_key: 'antigen_specific'                                              
INFO     spVIPESmulti: Labels: Using 'cluster_label' from adata.obs                                                
INFO     spVIPESmulti: --- Product of Experts (PoE) Configuration ---                                              
INFO     spVIPESmulti: Will use: Label-based PoE                                                                   


## 1. Load both trained models

In [8]:
group_indices_list = [list(map(int, g)) for g in adata_spv.uns["groups_obs_indices"]]

# Load baseline model
model_baseline = spVIPESmulti.model.spVIPESmulti.load("results/spvipes_bcells3", adata=adata_spv)

# Load recommended model (may not exist if malaria_bcells_recommended.ipynb hasn't been run yet)
try:
    model_rec = spVIPESmulti.model.spVIPESmulti.load("results/spvipes_bcells_recommended", adata=adata_spv)
    has_recommended = True
except FileNotFoundError:
    print("WARNING: results/spvipes_bcells_recommended not found.")
    print("Run malaria_bcells_recommended.ipynb first to generate the recommended model.")
    has_recommended = False

INFO     No backup URL provided for missing file results/spvipes_bcells3/model.pt                                  


ValueError: Failed to load model file at results/spvipes_bcells3/model.pt. If attempting to load a saved model from <v0.15.0, please use the util function `convert_legacy_save` to convert to an updated format.

## 2. Training curve comparison

Side-by-side training curves for the two runs.
Key metrics to compare:
- `reconstruction_loss_train/validation` — should decrease more smoothly in the recommended run
- `elbo_train/validation` — overall objective
- `kl_divergence_private_group_*` — should not spike in either run (BN fix already applied)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (model, label) in zip(axes, [
    (model_baseline, "Baseline (lr=1e-3, elbo_train scheduler, equal weights)"),
    (model_rec if has_recommended else model_baseline, "Recommended (lr=5e-4, recon_val scheduler, inv-freq weights)"),
]):
    history = model.history
    for key in ["reconstruction_loss_train", "reconstruction_loss_validation", "elbo_train", "elbo_validation"]:
        if key in history:
            vals = history[key]
            ax.plot(vals.index, vals[key], label=key, alpha=0.8)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=7, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Per-group KL comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
groups = ["CRXV (g0)", "NANP (g1)", "Njunc (g2)"]

for ax, (model, label) in zip(axes, [
    (model_baseline, "Baseline"),
    (model_rec if has_recommended else model_baseline, "Recommended"),
]):
    history = model.history
    for g, gname in enumerate(groups):
        key = f"kl_divergence_private_group_{g}_train"
        if key in history:
            vals = history[key]
            ax.plot(vals.index, vals[key], label=gname, alpha=0.8)
    ax.set_title(f"{label} — private KL per group")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("KL divergence")
    ax.legend()

plt.tight_layout()
plt.show()

## 3. UMAP comparison

Compute shared-latent UMAPs for both models and display side-by-side.
Good integration should show:
- Mixing across `antigen_specific` groups (CRXV, NANP, Njunc) in the shared UMAP
- Preservation of `cluster_label` structure

In [ ]:
# Compute embeddings and UMAPs for both models
results = {}

for model, tag in [(model_baseline, "baseline"), (model_rec if has_recommended else model_baseline, "recommended")]:
    adata_tmp = adata_spv.copy()
    latents = model.get_latent_representation(group_indices_list, batch_size=1024)
    spVIPESmulti.utils.store_latents(adata_tmp, latents, group_indices_list)
    spVIPESmulti.utils.compute_shared_umap(adata_tmp)
    results[tag] = adata_tmp

In [ ]:
color_keys = ["cluster_label", "antigen_specific", "time"]
n_cols = len(color_keys)
n_rows = 2  # baseline / recommended

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))

for row, (tag, label) in enumerate([("baseline", "Baseline"), ("recommended", "Recommended")]):
    adata_plot = results[tag]
    for col, color in enumerate(color_keys):
        ax = axes[row, col]
        sc.pl.umap(
            adata_plot,
            color=color,
            basis="umap_spvipesmulti_shared",
            title=f"{label} — {color}",
            show=False,
            ax=ax,
        )

plt.tight_layout()
plt.show()

## 4. Integration metrics

Quantitative comparison using available integration metrics.
- **iLISI**: integration LISI — higher means better mixing of `antigen_specific` groups
- **cLISI**: cell-type LISI — lower means better preservation of `cluster_label` structure
- **ASW** (silhouette): batch ASW for group mixing, label ASW for cluster separation

In [ ]:
# Integration metrics (if available)
try:
    metrics = {}
    for tag in ["baseline", "recommended"]:
        adata_m = results[tag]
        report = spVIPESmulti.metrics.integration_report(
            adata_m,
            embedding_key="X_spvm_shared",
            batch_key="antigen_specific",
            label_key="cluster_label",
        )
        metrics[tag] = report

    metric_df = pd.DataFrame(metrics).T
    print(metric_df.to_string())
except Exception as e:
    print(f"integration_report not available or errored: {e}")
    print("Compute metrics manually or check spVIPESmulti.metrics API.")

## 5. Summary table

In [ ]:
rows = []
for model, tag, label in [
    (model_baseline, "baseline", "Baseline"),
    (model_rec if has_recommended else model_baseline, "recommended", "Recommended"),
]:
    h = model.history
    def last(key):
        if key in h:
            vals = h[key]
            return float(vals[key].iloc[-1])
        return float("nan")
    rows.append({
        "Model": label,
        "Final epoch": len(h["reconstruction_loss_train"]) if "reconstruction_loss_train" in h else "?",
        "Final recon loss (train)": f"{last('reconstruction_loss_train'):.2f}",
        "Final recon loss (val)": f"{last('reconstruction_loss_validation'):.2f}",
        "Final ELBO (val)": f"{last('elbo_validation'):.2f}",
    })

pd.DataFrame(rows).set_index("Model")